# Prediction of cheating in films ~ The Feature Engineering

**Author**: Yacine Touileb et Nina Vivier Barte

**Date**: 23th November 2024


# 1. LIBRARY IMPORT AND FUNCTION CONSTRUCTION

In [2]:
!python -m spacy download en_core_web_md

# Computing libraries
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score
import string
import spacy
import torch
from transformers import BertModel, BertTokenizer

# Graphical libraries
import seaborn as sns
import matplotlib.pyplot as plt
## import dtreeviz

# Modelling libraries
from sklearn.linear_model import LogisticRegression # Logistic regression
from sklearn.ensemble import RandomForestClassifier # Random Forest
from sklearn.neural_network import MLPClassifier # Neural network

from google.colab import files
from IPython.display import display, Image
import io
import requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 12.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


#2. DATA IMPORT

In [3]:
# After running this cell, you need to load the dataset previously downloaded.
uploaded = files.upload()

Saving Letterboxd_annotations - Letterboxd_annotations.csv to Letterboxd_annotations - Letterboxd_annotations.csv


In [4]:
df = pd.read_csv(io.BytesIO(uploaded['Letterboxd_annotations - Letterboxd_annotations.csv']))
df = df[['description', 'txtCheat']]
df



,description,txtCheat
0,A cave man gets up in the morning to walk arou...,0
1,"Prince Salim, son of Akbar falls in love with ...",0
2,A young doctor receives an allotment for a sta...,0
3,"Quiet, unassuming Adam is changing in a major ...",0
4,A Camorra boss fakes his own death in order to...,0
...,...,...
2314,"During the pandemic, introverted painter Hélio...",0
2315,I really want to have sex before I turn 21! Ji...,0
2316,"Newlywed Jed and Angie this time, they have to...",1
2317,While working as an assistant to swindling dri...,0


# 3. FEATURE ENGINEERING

## 3.1 SPECIFIC BAG OF WORDS

In [5]:
cheating_words = [
    "cheat", "cheating", "betray", "betrayal", "lie", "lying", "affair",
    "unfaithful", "infidelity", "deception", "dishonesty", "adultery",
    "trust", "secrets", "manipulation", "hidden", "suspicion", "guilt"
]

# Function to count occurrences of specific words
def count_specific_words(sentence, words):
    sentence_words = sentence.lower().split()  # Convert to lowercase and split the words
    return {word: sentence_words.count(word) for word in words}

# Define `sentences` from the 'description' column
sentences = df['description'].dropna().tolist()

# Create a DataFrame containing counts of specific words
df_words = pd.DataFrame([count_specific_words(sentence, cheating_words) for sentence in sentences])

# Add the original column to the DataFrame for context
df_words['sentence'] = sentences

# Add the 'txtCheat' column to the df_words DataFrame
df_words['txtCheat'] = df['txtCheat'].values

# Reorganize the columns to place 'txtCheat' after 'sentence'
cols = ['sentence', 'txtCheat'] + [col for col in df_words.columns if col not in ['sentence', 'txtCheat']]
df_words = df_words[cols]

# Save the DataFrame as a CSV file
df_words.to_csv("baby_bagofwords.csv", index=False)

# Code to enable downloading in Colab
from google.colab import files
files.download("baby_bagofwords.csv")

df_words

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,sentence,txtCheat,cheat,cheating,betray,betrayal,lie,lying,affair,unfaithful,infidelity,deception,dishonesty,adultery,trust,secrets,manipulation,hidden,suspicion,guilt
0,A cave man gets up in the morning to walk arou...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,"Prince Salim, son of Akbar falls in love with ...",0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
2,A young doctor receives an allotment for a sta...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,"Quiet, unassuming Adam is changing in a major ...",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,A Camorra boss fakes his own death in order to...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2314,"During the pandemic, introverted painter Hélio...",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2315,I really want to have sex before I turn 21! Ji...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2316,"Newlywed Jed and Angie this time, they have to...",1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2317,While working as an assistant to swindling dri...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## 3.2 GENERAL BAG OF WORDS

In [6]:
# List of stopwords
stopwords = set([
    "a", "an", "the", "and", "or", "but", "if", "then", "so", "because", "as",
    "with", "on", "in", "at", "by", "for", "about", "against", "between",
    "to", "from", "up", "down", "out", "over", "under", "again", "further",
    "here", "there", "when", "where", "why", "how", "all", "any", "both",
    "each", "few", "more", "most", "other", "some", "such", "no", "nor",
    "not", "only", "own", "same", "so", "than", "too", "very"
])

# Function to clean a sentence (remove punctuation and stopwords)
def count_all_words_no_stopwords(sentence):
    if not isinstance(sentence, str):  # Check if the value is a string
        sentence = ""
    # Remove punctuation and convert to lowercase
    sentence = sentence.translate(str.maketrans('', '', string.punctuation)).lower()
    # Split into words
    words = sentence.split()
    # Remove stopwords
    filtered_words = [word for word in words if word not in stopwords]
    # Count the occurrences of each word
    return {word: filtered_words.count(word) for word in set(filtered_words)}


# Apply the function to the 'description' column
word_counts = [count_all_words_no_stopwords(sentence) for sentence in df['description']]

# Create a DataFrame for the words
df_words = pd.DataFrame(word_counts).fillna(0).astype(int)

# Add the original 'description' column
df_words['description'] = df['description']


# Calculate totals for each word in the DataFrame
word_totals = df_words.drop(columns=['description']).sum()

# Keep only words that appear at least IDKKKKKK times
words_to_keep = word_totals[word_totals >= 100].index.tolist()

# Filter the DataFrame to keep only these words
df_bagofwords = df_words[['description'] + words_to_keep]

# Add the 'txtCheat' column to the final DataFrame
df_bagofwords['txtCheat'] = df['txtCheat'].values

# Reorganize the columns to place 'txtCheat' after 'description'
cols = ['description', 'txtCheat'] + [col for col in df_bagofwords.columns if col not in ['description', 'txtCheat']]
df_bagofwords = df_bagofwords[cols]

# Save the result if necessary
df_bagofwords.to_csv("bagofwords.csv", index=False)

# Code to enable downloading in Colab
from google.colab import files
files.download("bagofwords.csv")

# Display a preview of the data
display(df_bagofwords)


<ipython-input-6-c548c937ec93>:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_bagofwords['txtCheat'] = df['txtCheat'].values


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,description,txtCheat,he,gets,after,of,she,him,man,girl,...,make,husband,city,can,becomes,three,through,during,people,work
0,A cave man gets up in the morning to walk arou...,0,3,1,2,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,"Prince Salim, son of Akbar falls in love with ...",0,1,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,A young doctor receives an allotment for a sta...,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,"Quiet, unassuming Adam is changing in a major ...",0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,A Camorra boss fakes his own death in order to...,0,1,0,1,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2314,"During the pandemic, introverted painter Hélio...",0,1,0,0,0,0,1,0,0,...,1,0,0,0,0,0,0,1,0,0
2315,I really want to have sex before I turn 21! Ji...,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2316,"Newlywed Jed and Angie this time, they have to...",1,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2317,While working as an assistant to swindling dri...,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 3.3 WORD EMBEDDINGS

In [7]:
nlp = spacy.load('en_core_web_md')  # Using spaCy model for word embeddings

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased') # Using BERT model for sentence embeddings
model = BertModel.from_pretrained('bert-base-uncased')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [8]:
def average_word_embedding(text):
    doc = nlp(text)
    word_embeddings = [token.vector for token in doc]
    if word_embeddings:
        return sum(word_embeddings) / len(word_embeddings)
    else:
        return [0] * 300  # spaCy uses 300-dimensional embeddings

df['Average Word Embedding'] = df['description'].apply(average_word_embedding)

In [9]:
# Expand the averaged word embeddings into separate columns
word_embedding_cols = pd.DataFrame(df['Average Word Embedding'].tolist(), index = df.index)
word_embedding_cols.columns = [f'Average_Word_Embedding_{i+1}' for i in range(300)]

In [10]:
# Concatenate the word embedding columns to the DataFrame
df = pd.concat([df, word_embedding_cols], axis = 1)

# Drop the temporary 'Average Word Embedding' column
df = df.drop(columns = ['Average Word Embedding'])

# Save the result if necessary
df.to_csv("word_embeddings.csv", index=False)

# Code to enable downloading in Colab
from google.colab import files
files.download("word_embeddings.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
pd.set_option('display.max_columns', None)
display(df)

,description,txtCheat,Average_Word_Embedding_1,Average_Word_Embedding_2,Average_Word_Embedding_3,Average_Word_Embedding_4,Average_Word_Embedding_5,Average_Word_Embedding_6,Average_Word_Embedding_7,Average_Word_Embedding_8,Average_Word_Embedding_9,Average_Word_Embedding_10,Average_Word_Embedding_11,Average_Word_Embedding_12,Average_Word_Embedding_13,Average_Word_Embedding_14,Average_Word_Embedding_15,Average_Word_Embedding_16,Average_Word_Embedding_17,Average_Word_Embedding_18,Average_Word_Embedding_19,Average_Word_Embedding_20,Average_Word_Embedding_21,Average_Word_Embedding_22,Average_Word_Embedding_23,Average_Word_Embedding_24,Average_Word_Embedding_25,Average_Word_Embedding_26,Average_Word_Embedding_27,Average_Word_Embedding_28,Average_Word_Embedding_29,Average_Word_Embedding_30,Average_Word_Embedding_31,Average_Word_Embedding_32,Average_Word_Embedding_33,Average_Word_Embedding_34,Average_Word_Embedding_35,Average_Word_Embedding_36,Average_Word_Embedding_37,Average_Word_Embedding_38,Average_Word_Embedding_39,Average_Word_Embedding_40,Average_Word_Embedding_41,Average_Word_Embedding_42,Average_Word_Embedding_43,Average_Word_Embedding_44,Average_Word_Embedding_45,Average_Word_Embedding_46,Average_Word_Embedding_47,Average_Word_Embedding_48,Average_Word_Embedding_49,Average_Word_Embedding_50,Average_Word_Embedding_51,Average_Word_Embedding_52,Average_Word_Embedding_53,Average_Word_Embedding_54,Average_Word_Embedding_55,Average_Word_Embedding_56,Average_Word_Embedding_57,Average_Word_Embedding_58,Average_Word_Embedding_59,Average_Word_Embedding_60,Average_Word_Embedding_61,Average_Word_Embedding_62,Average_Word_Embedding_63,Average_Word_Embedding_64,Average_Word_Embedding_65,Average_Word_Embedding_66,Average_Word_Embedding_67,Average_Word_Embedding_68,Average_Word_Embedding_69,Average_Word_Embedding_70,Average_Word_Embedding_71,Average_Word_Embedding_72,Average_Word_Embedding_73,Average_Word_Embedding_74,Average_Word_Embedding_75,Average_Word_Embedding_76,Average_Word_Embedding_77,Average_Word_Embedding_78,Average_Word_Embedding_79,Average_Word_Embedding_80,Average_Word_Embedding_81,Average_Word_Embedding_82,Average_Word_Embedding_83,Average_Word_Embedding_84,Average_Word_Embedding_85,Average_Word_Embedding_86,Average_Word_Embedding_87,Average_Word_Embedding_88,Average_Word_Embedding_89,Average_Word_Embedding_90,Average_Word_Embedding_91,Average_Word_Embedding_92,Average_Word_Embedding_93,Average_Word_Embedding_94,Average_Word_Embedding_95,Average_Word_Embedding_96,Average_Word_Embedding_97,Average_Word_Embedding_98,Average_Word_Embedding_99,Average_Word_Embedding_100,Average_Word_Embedding_101,Average_Word_Embedding_102,Average_Word_Embedding_103,Average_Word_Embedding_104,Average_Word_Embedding_105,Average_Word_Embedding_106,Average_Word_Embedding_107,Average_Word_Embedding_108,Average_Word_Embedding_109,Average_Word_Embedding_110,Average_Word_Embedding_111,Average_Word_Embedding_112,Average_Word_Embedding_113,Average_Word_Embedding_114,Average_Word_Embedding_115,Average_Word_Embedding_116,Average_Word_Embedding_117,Average_Word_Embedding_118,Average_Word_Embedding_119,Average_Word_Embedding_120,Average_Word_Embedding_121,Average_Word_Embedding_122,Average_Word_Embedding_123,Average_Word_Embedding_124,Average_Word_Embedding_125,Average_Word_Embedding_126,Average_Word_Embedding_127,Average_Word_Embedding_128,Average_Word_Embedding_129,Average_Word_Embedding_130,Average_Word_Embedding_131,Average_Word_Embedding_132,Average_Word_Embedding_133,Average_Word_Embedding_134,Average_Word_Embedding_135,Average_Word_Embedding_136,Average_Word_Embedding_137,Average_Word_Embedding_138,Average_Word_Embedding_139,Average_Word_Embedding_140,Average_Word_Embedding_141,Average_Word_Embedding_142,Average_Word_Embedding_143,Average_Word_Embedding_144,Average_Word_Embedding_145,Average_Word_Embedding_146,Average_Word_Embedding_147,Average_Word_Embedding_148,Average_Word_Embedding_149,Average_Word_Embedding_150,Average_Word_Embedding_151,Average_W